#### Build a Simple LLM Application with LCEL

In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models
- Using PromptTemplates and OutputParsers
- Using LangChain Expression Language (LCEL) to chain components together
- Debugging and tracing your application using LangSmith
- Deploying your application with LangServe

In [3]:
#Open AI and Open source models - Llama3, Gemma2,Mistral--Groq

import os
from dotenv import load_dotenv
load_dotenv()

import openai
#since we dont have openai paid, we can use gemini api free version
from google import genai
gemini_api_key=os.getenv('GEMINI_API_KEY')
groq_api_key=os.getenv('GROQ_API_KEY')

# groq_api_key

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

model=ChatGroq(model="llama-3.3-70b-versatile",groq_api_key=groq_api_key)
model

ChatGroq(output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000022172D5F9D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000022172DCC410>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [8]:
from langchain_core.messages import HumanMessage,SystemMessage

messages=[
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hello, How are you?")
]

res=model.invoke(messages)

In [9]:
res

AIMessage(content='Bonjour, comment allez-vous ?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 48, 'total_tokens': 56, 'completion_time': 0.037712651, 'completion_tokens_details': None, 'prompt_time': 0.002196354, 'prompt_tokens_details': None, 'queue_time': 0.337155616, 'total_time': 0.039909005}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ce7bc1685b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec0b0-7aad-7341-a5d6-0ecc1f7f3345-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 8, 'total_tokens': 56})

In [10]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(res)

'Bonjour, comment allez-vous ?'

In [12]:
#use the LCEL - to chain the components
chain=model|parser
chain.invoke(messages)

'Bonjour, comment allez-vous ?'

In [13]:
'''
    Instead of this, we can use one more technique, we can take a combination of user input and some application logic, 
    where i am also able to give the instruction, able to take the user input over there
    so, here, this application logic, which i'm actually using hee is to take the raw user input and transform into list of messages
    that are ready to passed to models
'''

#prompt template
from langchain_core.prompts import ChatPromptTemplate

generic_template="Translate the following into the {language}"

prompt=ChatPromptTemplate.from_messages(
    [
        ("system",generic_template),
        ("user","{text}")
    ]
)

prompt

ChatPromptTemplate(input_variables=['language', 'text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='Translate the following into the {language}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='{text}'), additional_kwargs={})])

In [17]:
# here, when you invoke, you can see the output as messages, which we gave above, but here, we're using this ChatPromptTemplate to convert to messages as below!!!
res=prompt.invoke({"language":"Telugu","text":"Hello world"})

In [18]:
res.to_messages()

[SystemMessage(content='Translate the following into the Telugu', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello world', additional_kwargs={}, response_metadata={})]

In [21]:
#now, trying to create a chain
#chaining components with LCEL

chain=prompt|model|parser
chain.invoke({"language":"Telugu","text":"Hello world!!!"})

'హలో ప్రపంచం!!!'